In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor, VotingRegressor
from sklearn.model_selection import train_test_split
import xgboost as xgb
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.impute import SimpleImputer

In [2]:
df = pd.read_csv("../cleanData.csv")

###  One-Hot Encoding for Month

In [3]:
month_dummies = pd.get_dummies(df['Month'], prefix='Month', drop_first=False)
df = pd.concat([df, month_dummies], axis=1)
df = df.drop(columns=['Month', 'Date', 'Year'])
df.drop(columns=['Month_sin', 'Month_cos'], inplace=True)

In [5]:
df.Year

0       2016
1       2007
2       2009
3       2013
4       2015
        ... 
6107    2010
6108    2011
6109    2012
6110    2014
6111    2008
Name: Year, Length: 6112, dtype: int64

# PCA

In [4]:
X = df.drop("Average_Bleaching", axis=1)
Y = df["Average_Bleaching"]

In [5]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

In [6]:
imputer = SimpleImputer(strategy='median')
X_train_imp = imputer.fit_transform(X_train_raw)
X_test_imp = imputer.transform(X_test_raw)

scaler = StandardScaler()
X_train_ss = scaler.fit_transform(X_train_imp)
X_test_ss = scaler.transform(X_test_imp)

In [7]:
pca = PCA(n_components=5)
X_train_pca = pca.fit_transform(X_train_ss)
X_test_pca = pca.transform(X_test_ss)

# OpTuna

In [8]:
import optuna
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.model_selection import cross_val_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

In [ ]:
def rf_objective(trial):

    model = RandomForestRegressor(
        n_estimators=trial.suggest_int("n_estimators", 50, 300),
        max_depth=trial.suggest_int("max_depth", 5, 40),
        min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 5),
        max_features=trial.suggest_categorical("max_features", ["sqrt", "log2"]),
        n_jobs=-1,
        random_state=42
    )

    pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        # ('selector', SelectKBest(score_func=f_regression,
        #                          k=trial.suggest_int("k_features", 5, X.shape[1]))),
        ('pca', PCA(
            n_components= 5 #trial.suggest_float("pca_variance", 0.90, 0.99)
        )),
        ('model', model)
    ])

    scores = cross_val_score(
        pipeline,
        X_train_raw,
        y_train,
        cv=4,
        scoring='r2',
        n_jobs=-1
    )

    return scores.mean()

rf_study = optuna.create_study(direction="maximize")

rf_study.optimize(rf_objective, n_trials=50)

print("========== RF BEST ==========")
print(rf_study.best_params)
print(f"Best R²: {rf_study.best_value:.4f}")

In [ ]:
def xgb_objective(trial):

    model = xgb.XGBRegressor(
        n_estimators=trial.suggest_int("n_estimators", 50, 400),
        learning_rate=trial.suggest_float("learning_rate", 0.005, 0.1),
        max_depth=trial.suggest_int("max_depth", 3, 15),
        subsample=trial.suggest_float("subsample", 0.5, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.5, 1.0),
        n_jobs=-1,
        random_state=42
    )

    pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        # ('selector', SelectKBest(
        #     score_func=f_regression,
        #     k=trial.suggest_int("k_features", 5, X.shape[1])
        # )),
        ('pca', PCA(
            n_components= 5 #trial.suggest_float("pca_variance", 0.90, 0.99)
        )),
        ('model', model)
    ])

    scores = cross_val_score(
        pipeline,
        X_train_raw,
        y_train,
        cv=4,
        scoring='r2',
        n_jobs=-1
    )

    return scores.mean()

xgb_study = optuna.create_study(direction="maximize")

xgb_study.optimize(xgb_objective, n_trials=50)

print("========== XGB BEST ==========")
print(xgb_study.best_params)
print(f"Best R²: {xgb_study.best_value:.4f}")

# Start

In [9]:
xgb_model = xgb.XGBRegressor(
    n_estimators=71, 
    learning_rate=0.03, 
    max_depth=14, 
    subsample=0.66, 
    colsample_bytree=0.95, 
    n_jobs=-1,
    random_state=42
)
# {'n_estimators': 71, 'learning_rate': 0.0324515787368114, 'max_depth': 14, 'subsample': 0.6625158076432023, 'colsample_bytree': 0.9518355944809992}


rf_model = RandomForestRegressor(
    n_estimators=265, 
    max_depth=21, 
    min_samples_leaf=1,
    max_features='log2',
    n_jobs=-1, 
    random_state=42
)
# {'n_estimators': 265, 'max_depth': 21, 'min_samples_leaf': 1, 'max_features': 'log2'}

In [11]:
voting = VotingRegressor([('xgb', xgb_model), ('rf', rf_model)])
voting.fit(X_train_pca, y_train)

# Evaluation
preds = voting.predict(X_test_pca)
r2 = r2_score(y_test, preds)
rmse = np.sqrt(mean_squared_error(y_test, preds))

print(f"\n--- Leakage-Free Results ---")
print(f"R-squared: {r2:.4f}")
print(f"RMSE: {rmse:.4f}")


--- Leakage-Free Results ---
R-squared: 0.3180
RMSE: 7.9990


# K-Fold

In [ ]:
from sklearn.model_selection import KFold, cross_val_score
import numpy as np

# %%
# K-Fold Validation 

n_splits = 4
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

r2_scores   = []
rmse_scores = []

for fold, (train_idx, test_idx) in enumerate(kf.split(X), 1):
    X_train_raw, X_test_raw = X.iloc[train_idx], X.iloc[test_idx]
    y_train,     y_test     = Y.iloc[train_idx], Y.iloc[test_idx]

    # Impute
    imputer     = SimpleImputer(strategy='median')
    X_train_imp = imputer.fit_transform(X_train_raw)
    X_test_imp  = imputer.transform(X_test_raw)

    # Scale
    scaler      = StandardScaler()
    X_train_ss  = scaler.fit_transform(X_train_imp)
    X_test_ss   = scaler.transform(X_test_imp)

    # PCA
    pca         = PCA(n_components=5)
    X_train_pca = pca.fit_transform(X_train_ss)
    X_test_pca  = pca.transform(X_test_ss)

    # Train
    voting      = VotingRegressor([('xgb', xgb_model), ('rf', rf_model)])
    voting.fit(X_train_pca, y_train)

    # Evaluate
    preds = voting.predict(X_test_pca)
    r2_scores.append(r2_score(y_test, preds))
    rmse_scores.append(np.sqrt(mean_squared_error(y_test, preds)))

    print(f"Fold {fold}  →  R²: {r2_scores[-1]:.4f}  |  RMSE: {rmse_scores[-1]:.4f}")

print(f"\n--- {n_splits}-Fold Summary ---")
print(f"R²   mean: {np.mean(r2_scores):.4f}  ±  {np.std(r2_scores):.4f}")
print(f"RMSE mean: {np.mean(rmse_scores):.4f}  ±  {np.std(rmse_scores):.4f}")